**Step 1 install the packages**

**Import the packages**

In [1]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from pinecone import Pinecone
from pinecone import Pinecone,ServerlessSpec
import os
from sentence_transformers import SentenceTransformer
from groq import Groq


C:\Users\suhan\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#!pip install Langchain
#!pip install pinecone
#!pip uninstall dotenv
!pip show dotenv

In [29]:
load_dotenv()
pinecone_api_key=os.getenv('apikey1')
groq_api = os.getenv("groq_key")

In [3]:
pinecone_api_key


'pcsk_52tfZr_9DVJqDMhhFgfg7Vk8DhdMJxkYyWZEfVJPFRAKLSQoPTiNEfwTZXcEtJXpoiQV5x'

**Load the pdf step 4**

In [4]:
loader=PyPDFLoader("budget_speech.pdf")

In [5]:
documents=loader.load()

In [ ]:
%pip install pypdf

In [ ]:
# documents

In [6]:
len(documents)

62

In [7]:
import os

In [8]:
os.listdir()

['.conda',
 '.env',
 '.vscode',
 'budget_speech.pdf',
 'myenv',
 'RAG_END_TO_END.ipynb']

In [9]:
print(documents[0].page_content)

GOVERNMENT OF INDIA
BUDGET 2024-2025
SPEECH
OF
NIRMALA SITHARAMAN
MINISTER OF FINANCE
July 23,  2024


**Split the text**

In [10]:
splitter=RecursiveCharacterTextSplitter(chunk_size=500,
                                       chunk_overlap=50)
chunks=splitter.split_documents(documents)

In [ ]:
# chunks

In [11]:
len(chunks)

234

**Convert to embeddings**

In [12]:
documents[0].page_content

'GOVERNMENT OF INDIA\nBUDGET 2024-2025\nSPEECH\nOF\nNIRMALA SITHARAMAN\nMINISTER OF FINANCE\nJuly 23,  2024'

In [13]:
documents[0].metadata

{'producer': 'Adobe PDF Library 24.2.159',
 'creator': 'Acrobat PDFMaker 24 for Word',
 'creationdate': '2024-07-23T00:18:17+05:30',
 'author': 'hss',
 'company': 'HCL Infosystems Limited',
 'moddate': '2024-07-23T00:30:42+05:30',
 'sourcemodified': 'D:20240722173931',
 'source': 'budget_speech.pdf',
 'total_pages': 62,
 'page': 0,
 'page_label': '1'}

In [ ]:
# chunks[0]

In [14]:
texts=[chunk.page_content for chunk in chunks]

In [ ]:
# texts

In [15]:
metadatas=[chunk.metadata for chunk in chunks]
metadatas

[{'producer': 'Adobe PDF Library 24.2.159',
  'creator': 'Acrobat PDFMaker 24 for Word',
  'creationdate': '2024-07-23T00:18:17+05:30',
  'author': 'hss',
  'company': 'HCL Infosystems Limited',
  'moddate': '2024-07-23T00:30:42+05:30',
  'sourcemodified': 'D:20240722173931',
  'source': 'budget_speech.pdf',
  'total_pages': 62,
  'page': 0,
  'page_label': '1'},
 {'producer': 'Adobe PDF Library 24.2.159',
  'creator': 'Acrobat PDFMaker 24 for Word',
  'creationdate': '2024-07-23T00:18:17+05:30',
  'author': 'hss',
  'company': 'HCL Infosystems Limited',
  'moddate': '2024-07-23T00:30:42+05:30',
  'sourcemodified': 'D:20240722173931',
  'source': 'budget_speech.pdf',
  'total_pages': 62,
  'page': 2,
  'page_label': '3'},
 {'producer': 'Adobe PDF Library 24.2.159',
  'creator': 'Acrobat PDFMaker 24 for Word',
  'creationdate': '2024-07-23T00:18:17+05:30',
  'author': 'hss',
  'company': 'HCL Infosystems Limited',
  'moddate': '2024-07-23T00:30:42+05:30',
  'sourcemodified': 'D:20240722

**converrt text to embeddings**

In [16]:
from sentence_transformers import SentenceTransformer
model=SentenceTransformer('all-MiniLM-L6-v2')
embeddings=model.encode(texts,
                       convert_to_numpy=True,
                       normalize_embeddings=True)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13629.90it/s]


In [17]:
embeddings.shape

(234, 384)

**Pinecone Connect**

In [18]:
from pinecone import Pinecone,ServerlessSpec
import os
pc=Pinecone(api_key=pinecone_api_key)
index=pc.Index('sentence-tra')
index

Index(host='https://sentence-tra-vqd6o4y.svc.aped-4627-b74a.pinecone.io')

In [19]:
index.describe_index_stats()

DescribeIndexStatsResponse(dimension=384, total_vector_count=0, metric='cosine', namespaces=0)

In [20]:
vectors=[]
for i in range(len(texts)):
    vectors.append(
        (
            str(i),
            embeddings[i].tolist(),
            {
                "text":texts[i],
                "page":metadatas[i].get("page",0)
            }
        )
    )

In [21]:
index.upsert(vectors=vectors)

UpsertResponse(upserted_count=234)

In [22]:
def retrive(query,top_k=5):
    query_vec=model.encode(
        query,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    results=index.query(
        vector=query_vec.tolist(),
        top_k=top_k,
        include_metadata=True
    )
    return results["matches"]

In [23]:
query="who is presenting the budget?"
results=retrive(query)

In [24]:
results

[ScoredVector(id='195', score=0.526123941, values=[], metadata={'page': 52, 'text': 'F. OTHERS \nThere are few other changes of minor nature. For details of the budget \nproposals, the Explanatory Memorandum and other relevant budget \ndocuments may be referred to.'}),
 ScoredVector(id='3', score=0.497145653, values=[], metadata={'page': 4, 'text': 'Budget 2024-2025 \n \nSpeech of \nNirmala Sitharaman \nMinister of Finance \nJuly 23, 2024 \nHon’ble Speaker,  \n I present the Budget for 2024-25.  \nIntroduction \n1. The people of India have reposed their faith in the government led by \nthe Hon’ble Prime Minister Shri Narendra Modi and re-elected it for a historic \nthird term under his leadership. We are grateful for their support, faith and \ntrust in our policies. We are determined to ensure that all Indians, regardless'}),
 ScoredVector(id='0', score=0.490199089, values=[], metadata={'page': 0, 'text': 'GOVERNMENT OF INDIA\nBUDGET 2024-2025\nSPEECH\nOF\nNIRMALA SITHARAMAN\nMINISTER 

In [25]:
print(index.describe_index_stats())


DescribeIndexStatsResponse(dimension=384, total_vector_count=234, metric='cosine', namespaces=1)


In [26]:
results[0]['metadata']['text']

'F. OTHERS \nThere are few other changes of minor nature. For details of the budget \nproposals, the Explanatory Memorandum and other relevant budget \ndocuments may be referred to.'

In [27]:
for r in results:
    print(r["score"])
    print(r["metadata"]["text"])
    print("-"*50)
    

0.526123941
F. OTHERS 
There are few other changes of minor nature. For details of the budget 
proposals, the Explanatory Memorandum and other relevant budget 
documents may be referred to.
--------------------------------------------------
0.497145653
Budget 2024-2025 
 
Speech of 
Nirmala Sitharaman 
Minister of Finance 
July 23, 2024 
Hon’ble Speaker,  
 I present the Budget for 2024-25.  
Introduction 
1. The people of India have reposed their faith in the government led by 
the Hon’ble Prime Minister Shri Narendra Modi and re-elected it for a historic 
third term under his leadership. We are grateful for their support, faith and 
trust in our policies. We are determined to ensure that all Indians, regardless
--------------------------------------------------
0.490199089
GOVERNMENT OF INDIA
BUDGET 2024-2025
SPEECH
OF
NIRMALA SITHARAMAN
MINISTER OF FINANCE
July 23,  2024
--------------------------------------------------
0.485088825
citizens. 
Budget Estimates 2024-25 
112. For the 

**LLM model prompt**

In [28]:
from groq import Groq


In [30]:
client=Groq(api_key=groq_api)


In [ ]:
# def generate_answer(context,question):
#     prompt=f"""
#     you are a helpful assistent.
#     use only the context below to answer.
#     Context:
#     {context}
#     Question:
#     {question}
#     Answer clearly and concisely.
#     """
#     response=client.char.completions.create(
#         model="
#     )
    

In [31]:
def generate_answer(context, question):

    prompt = f"""
You are a helpful assistant.

Use ONLY the context below to answer.

Context:
{context}

Question:
{question}

Answer clearly and concisely.
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content

**RAG pipeline**

In [32]:
def rag_pipeline(question):
    docs=retrive(question,top_k=5)
    context="\n\n".join(
        [doc["metadata"]["text"] for doc in docs]
    )
    answer=generate_answer(context,question)

    return answer

In [33]:
rag_pipeline("who is presenting the budget?")

'The budget is being presented by **Nirmala\u202fSitharaman, the Minister of Finance**.'